# Lesson 02 - Pagsusuri sa Microsoft Agent Framework

Ang **Microsoft Agent Framework (MAF)** ay isang pinag-isang balangkas para sa pagbuo ng mga AI agent. Nagbibigay ito ng malinis, na komponibleng arkitektura na may apat na pangunahing bahagi:

- **Client** – kumokonekta sa isang AI model endpoint at humahawak ng komunikasyon
- **Agent** – bumabalot sa isang client gamit ang mga tagubilin at mga kahulugan ng kasangkapan
- **Tools** – pinalalawak ang mga kakayahan ng agent gamit ang mga custom na function na maaaring tawagin ng modelo
- **Session** – nagpapanatili ng kasaysayan ng pag-uusap para sa mga multi-turn na interaksyon

Sa araling ito, gagawa tayo ng isang **travel booking agent** na nagche-check ng availability ng destinasyon gamit ang mga konseptong ito.


## Pagsasaayos


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Pag-unawa sa Arkitektura ng Agent Framework

Ang Microsoft Agent Framework ay sumusunod sa isang layered na arkitektura:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Kliyente** – Isang `AzureAIProjectAgentProvider` na kumokonekta sa isang Azure OpenAI deployment. Ito ang humahawak ng authentication, pag-format ng kahilingan, at pag-parse ng tugon.
2. **Ahente** – Ginawa mula sa kliyente sa pamamagitan ng `provider.create_agent()`, pinagsasama ng ahente ang pag-access sa modelo kasama ang mga tagubilin (system prompt) at mga kasangkapan.
3. **Mga Kasangkapan** – Mga function sa Python na may dekorasyon na `@tool` na maaaring tawagin ng ahente upang magsagawa ng mga aksyon o kumuha ng datos.
4. **Sesyon** – Isang `AgentSession` na object (ginawa gamit ang `agent.create_session()`) na nag-iimbak ng kasaysayan ng pag-uusap, na nagpapahintulot ng multi-turn na diyalogo kung saan naaalala ng ahente ang naunang konteksto.

Gawin nating hakbang-hakbang ang bawat layer.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Pagdaragdag ng mga Tool gamit ang @tool Decorator

Pinapayagan ng mga tool ang mga ahente na gumawa ng mga aksyon lampas sa paggawa ng teksto. Ang `@tool` decorator ay nagko-convert ng karaniwang Python function sa isang bagay na maaaring tawagan ng ahente.

Pangunahing puntos:
- Gumamit ng `Annotated[type, "description"]` upang maintindihan ng modelo ang bawat parameter.
- Ang docstring ang nagiging paglalarawan ng tool na nakikita ng modelo.
- Ang `approval_mode="never_require"` ay nangangahulugang ang tool ay tumatakbo nang awtomatiko nang walang kumpirmasyon mula sa user.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Paglikha ng isang Ahente gamit ang mga Kasangkapan

Ngayon, pinagsasama natin ang kliyente, mga tagubilin, at mga kasangkapan sa isang ahente. Ang `instructions` ay nagsisilbing prompt ng sistema — tinutukoy nito ang persona at pag-uugali ng ahente.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Maramihang Pag-uusap na Paikutin gamit ang Mga Session

Ang `AgentSession` (na nilikha sa pamamagitan ng `agent.create_session()`) ay nagtatala ng lahat ng mga mensahe sa isang pag-uusap. Sa pamamagitan ng pagpasa ng parehong session sa bawat tawag ng `agent.run()`, ang ahente ay may access sa buong kasaysayan ng pag-uusap at maaaring balikan ang mga naunang mensahe.

Pinapasa namin ang `tools=[check_destination_availability]` upang ang ahente ay maaaring tawagan ang aming tagasuri ng availability sa bawat pag-ikot.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Buod

Sa araling ito, sinaliksik mo ang apat na haligi ng Microsoft Agent Framework:

| Konsepto | Ano ang Iyong Natutunan |
|---------|-------------------------|
| **Kliyente** | `AzureAIProjectAgentProvider` ay kumokonekta sa Azure OpenAI gamit ang credential-based na awtentikasyon |
| **Ahente** | `provider.create_agent()` ay nagsasama ng koneksyon sa modelo kasama ang mga tagubilin at isang pangalan |
| **Mga Kasangkapan** | Ang `@tool` na decorator ay nagpapakita ng mga Python na function para tawagin ng ahente |
| **Sesyon** | `agent.create_session()` ay nagpapanatili ng kasaysayan ng pag-uusap sa maraming pagkakataon |

Ang mga pundasyong ito ay pinagsama upang lumikha ng mga ahenteng kayang magpanatili ng likas na pag-uusap, tumawag ng mga panlabas na function, at magpanatili ng konteksto — ang pundasyon para sa mas advanced na mga pattern ng ahente sa mga susunod na aralin.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pahayag ng Paunawa**:  
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI na pagsasalin na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagamat aming pinagsusumikapang maging tama ang salin, mangyaring tandaan na ang mga awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o di-tumpak na impormasyon. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang hindi pagkakaunawaan o maling interpretasyon na maaaring magmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
